# Kaggle / Colab runner

Clones the repo and runs the project scripts on a free GPU.

**Choose the phase** with `PHASE` in the first code cell: `3` = baseline with epoch-budget search, `4` = the baseline with 3 seeds plus the skin-tone analysis, `5` = the four image-processing settings (3 seeds each) and their comparison.

**Kaggle settings (right-hand panel):** Accelerator → *GPU T4 x2* or *GPU P100*; Internet → *On*
(needs a phone-verified account).

**Run it with *Save Version → Save & Run All (Commit)*,** not interactively. That runs in the background (up to 12 h, no idle timeout) and keeps everything in `/kaggle/working` as the version's output.

The repo and data live in a temporary folder (`/kaggle/temp`), so the ~3 GB of images are not saved as notebook output.
Each finished budget trial and CV fold is saved to `/kaggle/working/baseline_folds` (Phase 3) `/kaggle/working/seed_folds` (Phase 4) or `/kaggle/working/phase5_folds/<setting>` (Phase 5) straight away, so a crash or timeout keeps the finished folds. To resume, add the previous version's output as an input and set `RESUME_FROM` below; the training script skips the folds already saved.

Results are zipped into `/kaggle/working/phase<PHASE>_outputs.zip` at the end: download it and put it in the repo root.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/sehersharik/skin-tone-triage.git"
WORK = "/kaggle/temp" if os.path.isdir("/kaggle") else "/content"   # Kaggle or Colab
os.makedirs(WORK, exist_ok=True)
OUTPUT = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"  # survives a timeout on Kaggle
PHASE = 5   # 3 = baseline + epoch budget, 4 = 3 seeds + skin-tone analysis, 5 = image-processing settings
FOLD_DIR = f"{OUTPUT}/" + {3: "baseline_folds", 4: "seed_folds", 5: "phase5_folds"}[PHASE]
RESUME_FROM = None   # e.g. "/kaggle/input/<previous-version-output>/seed_folds"
if RESUME_FROM and not os.path.isdir(FOLD_DIR):
    !cp -r {RESUME_FROM} {FOLD_DIR}
%cd {WORK}
!rm -rf skin-tone-triage
if subprocess.run(["git", "clone", "-q", REPO_URL]).returncode != 0:
    raise RuntimeError("Could not clone the repo. On Kaggle: Settings -> Internet -> On "
                       "(needs a phone-verified account), then run again.")
%cd skin-tone-triage
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data: metadata, splits, and only the images we use

`build_metadata` stops if the splits differ from the committed `reports/split_fingerprint.txt`.

In [ ]:
!python -m src.download_scin --config configs/data.yaml --images none
!python -m src.build_metadata --config configs/labels.yaml
!python -m src.download_scin --config configs/data.yaml --images used
!python -m src.prepare_images --config configs/baseline.yaml

## Phase 3: baseline 5-fold CV (test set untouched) and evaluation

In [ ]:
if PHASE == 3:
    !python -m src.train --config configs/baseline.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.evaluate --config configs/baseline.yaml

## Phase 4: 3 seeds on the same folds, then the skin-tone analysis (test set untouched)

About 4 min of training per seed on a T4. `image_factors` measures brightness, contrast, sharpness and resolution of every CV photo; `tone_analysis` writes `tone_analysis_cv` to `reports/metrics.json` plus the `phase4_*` figures.

In [ ]:
if PHASE == 4:
    !python -m src.train --config configs/seeds.yaml --fold-outputs-dir {FOLD_DIR}
    !python -m src.image_factors --config configs/seeds.yaml
    !python -m src.tone_analysis --config configs/seeds.yaml

## Phase 5: image-processing settings, 3 seeds x 5 folds each (test set untouched)

For each setting the epoch budget is re-picked on fold 1 (seed 42), then 3 seeds x 5 folds are trained. Settings run one after another; every finished budget trial and fold is saved, so a rerun with `RESUME_FROM` continues where it stopped. `compare_settings` writes `phase5_cv` to `reports/metrics.json` and the `phase5_*` figures.

In [ ]:
if PHASE == 5:
    for setting in ["none", "standard_aug", "shades_of_gray", "clahe"]:
        !python -m src.train --config configs/phase5/{setting}.yaml --fold-outputs-dir {FOLD_DIR}/{setting}
    !python -m src.compare_settings --config configs/phase5/compare.yaml

In [ ]:
!pytest -q -p no:warnings
OUT = f"{OUTPUT}/phase{PHASE}_outputs.zip"
!zip -qr {OUT} reports docs/figures runs
import glob, json
for log in sorted(glob.glob({3: "reports/baseline_train_log.json", 4: "reports/seeds/*_train_log.json",
                                5: "reports/phase5/*_train_log.json"}[PHASE])):
    t = json.load(open(log))["timing"]
    print(log, "| GPU:", t["gpu"], "| seconds per CV fold:", t["seconds_per_cv_fold"])
print("Download:", OUT)